# Notebook 4 — Context Assembly & Gemini 1.5 Pro Generation
**Layer:** Generation · **Scope:** Dual-path retrieval → fused context → Gemini 1.5 Pro answer  
**Inputs:** Cypher result (NB3) · FAISS vector index (NB2) · `feature_metadata_20260403.json`  
**Outputs:** Structured JSON answer with price estimate, comparables, factors, caveats

## 4.1 Install & import dependencies

In [1]:
import json
import re
import pickle
import numpy as np
import faiss
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize
from neo4j import GraphDatabase
import google.genai as genai          # new SDK (google-genai v1.72, already installed)
from google.genai import types
import os
from dotenv import load_dotenv

FEATURE_DIR = Path("02_feature_layer/training/outputs")
INDEX_DIR   = Path("03_vector_index")
META_PATH   = FEATURE_DIR / "feature_metadata_20260403.json"

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
#FEATURE_DIR = Path("02_feature_layer/training/outputs")
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
INDEX_DIR   = ROOT / '05_llm_layer' / '03_vector_index'
META_PATH   = FEATURE_DIR / "feature_metadata_20260412.json"


# ── Load EMBED_DIM from NB02 config (256 for TF-IDF/SVD) ──────────────────
with open(INDEX_DIR / "embed_config.json") as f:
    cfg = json.load(f)
EMBED_DIM = cfg["embed_dim"]   # 256
print(f"EMBED_DIM loaded from embed_config.json: {EMBED_DIM}")

# ── Load sklearn embed pipeline from NB02 ─────────────────────────────────
with open(INDEX_DIR / "tfidf_svd_pipeline.pkl", "rb") as f:
    embed_pipeline = pickle.load(f)
print("embed_pipeline loaded from tfidf_svd_pipeline.pkl")

# ── Gemini clients — new SDK pattern ──────────────────────────────────────
ENV_PATH = ROOT / '.env'

# Load the file from that specific path
load_dotenv(dotenv_path=ENV_PATH)

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = "your_api_key_here"   # <-- update before running
FLASH_MODEL    = "gemini-2.0-flash"    # stable on v1 endpoint
PRO_MODEL      = "gemini-1.5-pro"

client = genai.Client(
    api_key      = GEMINI_API_KEY,
    http_options = types.HttpOptions(api_version="v1")  # avoids v1beta 404
)
print(f"Gemini client ready (Flash: {FLASH_MODEL}, Pro: {PRO_MODEL})")

# ── Neo4j connection ───────────────────────────────────────────────────────
NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"    # <-- update before running

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# ── Price drift constants from README section 3 ───────────────────────────
TRAIN_MEAN_PRICE = 468788
TEST_MEAN_PRICE  = 614711
TEMPORAL_SPLIT   = 2023

print("All clients ready.")

EMBED_DIM loaded from embed_config.json: 256
embed_pipeline loaded from tfidf_svd_pipeline.pkl
Gemini client ready (Flash: gemini-2.0-flash, Pro: gemini-1.5-pro)
All clients ready.


## 4.2 Load vector indexes and metadata

In [2]:
index_pre  = faiss.read_index(str(INDEX_DIR / "index_pre2023.faiss"))
index_post = faiss.read_index(str(INDEX_DIR / "index_post2023.faiss"))
meta_pre   = pd.read_parquet(INDEX_DIR / "meta_pre2023.parquet")
meta_post  = pd.read_parquet(INDEX_DIR / "meta_post2023.parquet")

with open(META_PATH) as f:
    feature_meta = json.load(f)

print(f"pre-2023  index : {index_pre.ntotal:,} vectors  (dim={index_pre.d})")
print(f"post-2023 index : {index_post.ntotal:,} vectors  (dim={index_post.d})")
print(f"EMBED_DIM check : index.d={index_pre.d} == cfg EMBED_DIM={EMBED_DIM} -> {index_pre.d == EMBED_DIM}")

pre-2023  index : 178,589 vectors  (dim=256)
post-2023 index : 82,110 vectors  (dim=256)
EMBED_DIM check : index.d=256 == cfg EMBED_DIM=256 -> True


## 4.3 Import routing helpers from Notebook 3

In [3]:
# Routing helpers — re-declared here so NB04 is self-contained.
# In production: refactor into hdb_router.py and import.

VALID_TOWNS = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN"
]
VALID_FLAT_TYPES = ["1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"]

CLASSIFIER_SYSTEM = """You are a query classifier for an HDB flat database system.

Classify the user query into EXACTLY ONE of these intent types:
- PRICE_ESTIMATION: user wants a price estimate or valuation
- NEIGHBOURHOOD: user wants nearby amenities (MRT, malls, food courts, highways)
- SCHOOL_CATCHMENT: user wants school proximity or quality
- INVESTMENT_TEMPORAL: user wants historical price trends or town-level appreciation
- LEASE_ADVISORY: user wants guidance related to lease remaining years

Extract these slots if mentioned (null if not mentioned):
- town: must be one of the 26 valid HDB towns (uppercase)
- flat_type: one of [1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION]
- room_count: integer 1-6
- budget_sgd_min: integer
- budget_sgd_max: integer
- year_min: integer 2015-2026
- year_max: integer 2015-2026
- lease_years_max: integer

Respond ONLY with valid JSON. No explanation, no markdown fences.
Format: {"intent": "...", "slots": {"town": null, "flat_type": null, ...}}
"""

CYPHER_TEMPLATES = {
    "PRICE_ESTIMATION": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
  AND ($room_count IS NULL OR f.room_count = $room_count)
  AND ($budget_sgd_min IS NULL OR f.resale_price >= $budget_sgd_min)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
RETURN percentileCont(f.resale_price,0.5) AS median_price,
       avg(f.resale_price) AS avg_price, stDev(f.resale_price) AS std_price,
       count(f) AS tx_count, avg(f.lease_remaining_years) AS avg_lease,
       avg(f.floor_area_sqm) AS avg_area,
       min(f.transaction_year) AS year_min, max(f.transaction_year) AS year_max""",

    "NEIGHBOURHOOD": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
RETURN t.name AS town, avg(f.dist_to_mrt_m) AS avg_mrt_dist_m,
       avg(f.dist_to_foodcourt_m) AS avg_foodcourt_dist_m,
       avg(f.dist_to_nearest_mall_m) AS avg_mall_dist_m,
       avg(f.mall_count_3km) AS avg_mall_count_3km,
       avg(f.mall_weighted_access_3km) AS avg_mall_access_score,
       avg(f.dist_to_highway_m) AS avg_highway_dist_m, count(f) AS flat_count""",

    "SCHOOL_CATCHMENT": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
RETURN t.name AS town,
       avg(f.primary_school_quality_1km_weighted) AS avg_school_quality,
       avg(f.primary_school_top_quality_1km) AS avg_top_school_quality,
       avg(f.school_count_1km) AS avg_schools_in_1km,
       avg(f.dist_to_nearest_school_m) AS avg_school_dist_m,
       avg(f.resale_price) AS avg_price, count(f) AS flat_count
ORDER BY avg_school_quality DESC""",

    "INVESTMENT_TEMPORAL": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($year_min IS NULL OR f.transaction_year >= $year_min)
  AND ($year_max IS NULL OR f.transaction_year <= $year_max)
RETURN t.name AS town, f.transaction_year AS year,
       avg(f.resale_price) AS avg_price,
       percentileCont(f.resale_price,0.5) AS median_price,
       count(f) AS tx_count
ORDER BY t.name, f.transaction_year""",

    "LEASE_ADVISORY": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($lease_years_max IS NULL OR f.lease_remaining_years <= $lease_years_max)
RETURN CASE
         WHEN f.lease_remaining_years < 50 THEN '<50 years'
         WHEN f.lease_remaining_years < 70 THEN '50-69 years'
         ELSE '70+ years'
       END AS lease_band,
       avg(f.resale_price) AS avg_price,
       percentileCont(f.resale_price,0.5) AS median_price,
       count(f) AS tx_count
ORDER BY lease_band"""
}


def classify_query(user_query: str) -> dict:
    """
    Classify user query using Gemini Flash via new SDK.
    Old pattern: flash.generate_content(prompt)
    New pattern: client.models.generate_content(model=FLASH_MODEL, contents=prompt)
    """
    prompt   = CLASSIFIER_SYSTEM + "\n\nQuery: " + user_query
    response = client.models.generate_content(
        model    = FLASH_MODEL,
        contents = prompt
    )
    raw = response.text.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)


def slots_to_params(slots: dict) -> dict:
    town      = slots.get("town")
    flat_type = slots.get("flat_type")
    if town and town.upper() not in VALID_TOWNS:
        town = None
    if flat_type and flat_type.upper() not in VALID_FLAT_TYPES:
        flat_type = None
    return {
        "town":            town.upper() if town else None,
        "flat_type":       flat_type.upper() if flat_type else None,
        "room_count":      slots.get("room_count"),
        "budget_sgd_min":  slots.get("budget_sgd_min"),
        "budget_sgd_max":  slots.get("budget_sgd_max"),
        "year_min":        slots.get("year_min"),
        "year_max":        slots.get("year_max"),
        "lease_years_max": slots.get("lease_years_max"),
    }


print("Routing helpers defined (classify_query, slots_to_params, CYPHER_TEMPLATES).")

Routing helpers defined (classify_query, slots_to_params, CYPHER_TEMPLATES).


## 4.4 Vector similarity retrieval function

In [4]:
def embed_query(query_text: str) -> np.ndarray:
    """
    Embed a single query string using the fitted TF-IDF + SVD pipeline
    saved by NB02 (embed_pipeline.pkl).
    Returns L2-normalised float32 array of shape (1, EMBED_DIM).
    """
    vec = embed_pipeline.transform([query_text]).astype(np.float32)
    vec = normalize(vec, norm="l2").astype(np.float32)
    return vec   # shape (1, 256)


def vector_retrieve(query_text: str, is_recent: bool, top_k: int = 20) -> pd.DataFrame:
    """
    Embed the query and retrieve top-k comparable flats.
    Uses post-2023 index for current-price queries, pre-2023 for historical.
    """
    q_vec = embed_query(query_text)
    faiss.normalize_L2(q_vec)

    if is_recent:
        D, I = index_post.search(q_vec, top_k)
        meta  = meta_post
    else:
        D, I = index_pre.search(q_vec, top_k)
        meta  = meta_pre

    hits = meta.iloc[I[0]].copy()
    hits["similarity_score"] = D[0]
    return hits.reset_index(drop=True)


print("embed_query() and vector_retrieve() defined.")

embed_query() and vector_retrieve() defined.


## 4.5 Context assembly — merge graph + vector + metadata

In [5]:
FACTOR_MAP = {
    "PRICE_ESTIMATION":   ["level_mid","lease_remaining_years","floor_area_sqm",
                            "room_count","dist_to_mrt_m","orientation_score"],
    "NEIGHBOURHOOD":      ["dist_to_mrt_m","dist_to_highway_m","dist_to_foodcourt_m",
                            "mall_count_3km","mall_weighted_access_3km"],
    "SCHOOL_CATCHMENT":   ["dist_to_nearest_school_m","school_count_1km",
                            "primary_school_quality_1km_weighted",
                            "primary_school_top_quality_1km","primary_school_count_1km"],
    "INVESTMENT_TEMPORAL":["transaction_year","resale_price"],
    "LEASE_ADVISORY":     ["lease_remaining_years","resale_price"],
}

def get_feature_definitions(intent: str) -> str:
    """Pull only relevant factor definitions from feature_metadata_20260403.json."""
    relevant = FACTOR_MAP.get(intent, [])
    lines = []
    for feat in relevant:
        if feat in feature_meta.get("features", {}):
            d = feature_meta["features"][feat]
            lines.append(f"  {feat}: {d.get('description','')}")
    return "\n".join(lines) if lines else "(see feature_metadata_20260403.json)"

def assemble_context(
    user_query: str,
    intent: str,
    cypher_records: list,
    vector_hits: pd.DataFrame,
    is_recent: bool
) -> str:
    """
    Build the structured context string for Gemini 1.5 Pro.
    Sections:
      1. GRAPH AGGREGATES  — Cypher results
      2. COMPARABLE TRANSACTIONS — top-5 from vector store
      3. FEATURE DEFINITIONS — from feature_metadata_20260403.json
      4. TEMPORAL CONTEXT — price drift warning if post-2023
    """
    ctx = []

    # 1. Graph aggregates
    ctx.append("=== GRAPH AGGREGATES (from Neo4j) ===")
    ctx.append(json.dumps(cypher_records[:10], indent=2, default=str))

    # 2. Top-5 comparable transactions
    ctx.append("\n=== COMPARABLE TRANSACTIONS (top-5 from vector index) ===")
    COMP_COLS = ["address_key","town","flat_type","floor_area_sqm",
                 "level_mid","lease_remaining_years","resale_price","transaction_year"]
    top5 = vector_hits.head(5)[COMP_COLS]
    ctx.append(top5.to_string(index=False))

    # 3. Feature definitions
    ctx.append("\n=== FEATURE DEFINITIONS (from feature_metadata_20260403.json) ===")
    ctx.append(get_feature_definitions(intent))

    # 4. Temporal context — price drift warning
    if is_recent:
        ctx.append(
            "\n=== TEMPORAL CONTEXT ===\n"
            f"Query uses post-2023 transactions. "
            f"Mean price post-2023: ${TEST_MEAN_PRICE:,} vs pre-2023: ${TRAIN_MEAN_PRICE:,} "
            f"(31% higher — Singapore property appreciation 2023-2026). "
            f"Reflect this in any price estimate caveats."
        )

    return "\n".join(ctx)

print("assemble_context() defined.")

assemble_context() defined.


## 4.6 Gemini 1.5 Pro generation call

In [6]:
PRO_SYSTEM = """You are an expert HDB property advisor for Singapore.

RULES (strictly enforced):
1. Answer ONLY using data in GRAPH AGGREGATES and COMPARABLE TRANSACTIONS sections.
2. Do NOT introduce any price figures, school names, MRT station names, or locations
   not present in the provided context.
3. Cite the transaction_year range of the comparables you used.
4. If post-2023 data is used, include the price drift caveat in your response.
5. Output ONLY valid JSON matching the schema below. No markdown, no explanation.

OUTPUT SCHEMA:
{
  "estimate_sgd": <integer or null>,
  "low_sgd":      <integer or null>,
  "high_sgd":     <integer or null>,
  "basis_count":  <integer>,
  "key_factors": [
    {"factor_name": <string>, "value": <string>, "impact": <string>}
  ],
  "caveats": [<string>],
  "comparable_years_range": "<year_min>-<year_max>",
  "narrative": "<2-3 sentence plain English summary>"
}
"""


def generate_answer(user_query: str, context: str) -> dict:
    """
    Call Gemini 1.5 Pro via new SDK (google-genai v1.72).
    Old pattern: pro.generate_content(prompt)
    New pattern: client.models.generate_content(model=PRO_MODEL, contents=prompt)
    """
    prompt   = PRO_SYSTEM + "\n\n" + context + "\n\nUSER QUERY: " + user_query
    response = client.models.generate_content(
        model    = PRO_MODEL,
        contents = prompt
    )
    raw = response.text.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)


print("generate_answer() defined.")

generate_answer() defined.


## 4.7 Full pipeline — single query end-to-end

In [7]:
def is_recent_query(slots: dict) -> bool:
    """Decide which index slice to use based on slot year range."""
    year_min = slots.get("year_min")
    year_max = slots.get("year_max")
    if year_min and year_min >= TEMPORAL_SPLIT:
        return True
    if year_max and year_max >= TEMPORAL_SPLIT:
        return True
    return False   # default to pre-2023 for unspecified queries

def run_pipeline(user_query: str) -> dict:
    # Step 1 — Route
    classified = classify_query(user_query)   # from NB3
    intent     = classified["intent"]
    slots      = classified["slots"]
    params     = slots_to_params(slots)       # from NB3
    cypher     = CYPHER_TEMPLATES[intent]     # from NB3
    recent     = is_recent_query(slots)

    # Step 2 — Dual-path retrieval
    with driver.session() as session:
        cypher_records = [dict(r) for r in session.run(cypher, **params)]

    vector_hits = vector_retrieve(user_query, is_recent=recent, top_k=20)

    # Step 3 — Assemble context
    context = assemble_context(user_query, intent, cypher_records, vector_hits, recent)

    # Step 4 — Generate
    answer = generate_answer(user_query, context)

    return {
        "query":       user_query,
        "intent":      intent,
        "slots":       slots,
        "is_recent":   recent,
        "cypher_rows": len(cypher_records),
        "vector_hits": len(vector_hits),
        "answer":      answer
    }

# Test with one query
result = run_pipeline("How much is a 4-room flat in Bishan worth today?")
print(json.dumps(result["answer"], indent=2))

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 41.478984524s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

## 4.8 Batch test — all 5 intent types

In [ ]:
test_queries = [
    "How much is a 4-room flat in Bishan worth?",
    "What amenities are near Ang Mo Kio flats?",
    "Which areas have the best primary schools under $700K?",
    "Which town had the best price growth from 2020 to 2025?",
    "Should I buy a flat with only 55 years of lease left?",
]

for q in test_queries:
    r = run_pipeline(q)
    print(f"Intent: {r['intent']}")
    print(f"  Cypher rows: {r['cypher_rows']} | Vector hits: {r['vector_hits']} | Recent: {r['is_recent']}")
    ans = r["answer"]
    print(f"  Estimate: ${ans.get('estimate_sgd','N/A'):,}" if ans.get("estimate_sgd") else f"  Narrative: {ans.get('narrative','')[:80]}")
    print(f"  Caveats: {ans.get('caveats', [])}")
    print()

driver.close()
print("Notebook 4 complete.")